# Steam Reviews Sub-sample Logistic Regression
direct link to download data:
https://drive.google.com/uc?id=1tWJSm0wWvduyNjx0Mnk6NBpeFahsZNbI&export=download

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Optional utilities (only if you actually use them)
import ast
import json

In [2]:
# INITIALIZING THE SPARK

spark = (
    SparkSession.builder
    .appName("steam_games_set")
    .config("spark.driver.memory", "4g")
    .config("spark.driver.cores", "1")
    .config("spark.executor.instances", "8")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "10g")
    .getOrCreate()
)

In [3]:
# Loading up our parquet
# LOAD THE DATA FROM WHEREVER YOU ARE HOLDING IT!
df = spark.read.parquet('cleaned_sampled.parquet')

df.printSchema()

# df.show(5, truncate=False)

root
 |-- author_steamid: long (nullable = true)
 |-- appid: long (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: timestamp_ntz (nullable = true)
 |-- review: string (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- written_during_early_access: boolean (nullable = true)
 |-- language: string (nullable = true)
 |-- timestamp_created: timestamp_ntz (nullable = true)
 |-- timestamp_updated: timestamp_ntz (nullable = true)



In [4]:
# ===============================
# Keep English reviews only
# ===============================

df_english = (
    df
    .filter(F.col('language') == 'english')
)

In [5]:
# ===============================
# Define churn / retention label
# ===============================
# retained = 1 means the player came back
# retained = 0 means likely churned

df_labeled = (
    df_english
    .filter(F.col('author_playtime_last_two_weeks').isNotNull())
    .withColumn(
        'retained',
        F.when(F.col('author_playtime_last_two_weeks') > 60, 1).otherwise(0)
    )
)

### Retention / Churn Heuristic

For this initial baseline model, a player is considered **retained** if they recorded more than 60 minutes of playtime within the last two weeks. Players with 60 minutes or less are treated as likely churned users.

The 60-minute threshold acts as a practical behavioral heuristic rather than a definitive business rule. The assumption is that players who return and spend at least one hour actively engaging with a game over a recent two-week period demonstrate meaningful continued interest and engagement. In contrast, very low or zero recent playtime may indicate disengagement, abandonment, or temporary inactivity.

This threshold was intentionally chosen as a lightweight and interpretable starting point for experimentation. It helps transform continuous playtime behavior into a binary classification problem suitable for Logistic Regression while remaining easy to explain from a business perspective.

Future iterations of the project may refine this definition using:
- percentile-based engagement thresholds,
- genre-specific activity expectations,
- rolling activity windows,
- survival analysis,
- or clustering methods to identify natural retention breakpoints.

In [6]:
# ===============================
# Check label distribution
# ===============================

df_labeled.groupBy('retained').count().show()

+--------+------+
|retained| count|
+--------+------+
|       1| 58241|
|       0|888660|
+--------+------+



In [7]:
# ===============================
# Select basic model features
# ===============================

feature_cols = [
    'author_num_games_owned',
    'author_num_reviews',
    'author_playtime_forever',
    'author_playtime_at_review',
    'votes_up',
    'comment_count',
    'weighted_vote_score'
]

df_model = (
    df_labeled
    .select(feature_cols + ['retained'])
    .dropna()
)

In [8]:
# ===============================
# Assemble features
# ===============================

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features'
)

df_features = (
    assembler
    .transform(df_model)
    .select('features', 'retained')
)

In [9]:
# ===============================
# Train / test split
# ===============================

train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

In [10]:
# ===============================
# Train logistic regression
# ===============================

lr = LogisticRegression(
    featuresCol='features',
    labelCol='retained'
)

lr_model = lr.fit(train_df)

In [11]:
# ===============================
# Make predictions
# ===============================

predictions = lr_model.transform(test_df)

predictions.select(
    'retained',
    'prediction',
    'probability'
).show(10, truncate=False)

+--------+----------+-----------------------------------------+
|retained|prediction|probability                              |
+--------+----------+-----------------------------------------+
|0       |0.0       |[0.9459949861217873,0.05400501387821266] |
|0       |0.0       |[0.9454354187244816,0.05456458127551844] |
|0       |0.0       |[0.9562552985945252,0.04374470140547482] |
|0       |0.0       |[0.9448187092922177,0.05518129070778233] |
|0       |0.0       |[0.9462477548515351,0.05375224514846488] |
|0       |0.0       |[0.9482530666046202,0.051746933395379835]|
|0       |0.0       |[0.9553342753956707,0.04466572460432927] |
|0       |0.0       |[0.9450766639035791,0.05492333609642086] |
|0       |0.0       |[0.9451023968081885,0.05489760319181147] |
|0       |0.0       |[0.9510252301381374,0.048974769861862555]|
+--------+----------+-----------------------------------------+
only showing top 10 rows



In [12]:
# ===============================
# Evaluate with AUC
# ===============================

evaluator = BinaryClassificationEvaluator(
    labelCol='retained',
    rawPredictionCol='rawPrediction',
    metricName='areaUnderROC'
)

auc = evaluator.evaluate(predictions)

print(f'AUC: {auc}')

AUC: 0.8184273681699763


In [13]:
# ===============================
# Quick accuracy check
# ===============================

accuracy = (
    predictions
    .filter(F.col('retained') == F.col('prediction'))
    .count()
    / predictions.count()
)

print(f'Accuracy: {accuracy}')

Accuracy: 0.9397471783773487
